# 05 — Evaluate everything (both arms + real DetectorChannel + real-detector Docket)

The merged evaluation session (~4–5 h GPU): everything downstream of notebook
04's weights, in one place. Attach **04's output** and the **DENTEX dataset**
as data sources before running.

1. **Real detection metrics per arm, across degradation severities** — the
   mAP / per-class-F1 numbers `src/eval/metrics.py` has been ready for since
   Phase 4, plus the severity sweep notebook 03 described but left as prose.
   The severity table IS the robustness headline: the claim is the
   *difference* between the arms' curves, not either curve alone.
2. **`DetectorChannel` against a real detector, for the first time** — wires
   the trained checkpoint + arm-matched confidence head into
   `src/models/detector_channel.py` (until now only exercised against fakes
   in `tests/test_detector_channel.py`).
3. **The Docket with a real trained detector** — E6's exact protocol
   (calibrate on held-out first shots → run all policy arms → leaderboard)
   with `DetectorChannel` in place of E6's linear `RealImageChannel`. This is
   the "E3-type leaderboard on a real model" upgrade
   `docs/kaggle_instructions.md` step 6 asks for. Deliberately NOT done by
   rerunning `experiments.run_all e3` — `build_world()` and its
   `calibrate_separation`/`calibrate_loss_scale` mutate surrogate-only knobs
   (an anchored analytic reader), which a real detector doesn't have. E6 is
   the repo's established path for a real channel; this follows it verbatim.

**Honest scope**: (1) generalizes notebook 03's run-verified inference loop;
(2) and (3) are new glue, AST-checked by the preflight but first executed
here. Two caveats to carry into any writeup: the confidence head was trained
on full 800×800 radiographs while the channel feeds it rendered tooth crops
(a distribution shift — the head's crop-level readings are themselves a
result, not an assumption); and the Docket's calibration here is fit fresh
on this channel's own held-out first shots, which is exactly the deployment
precondition E14 showed is load-bearing.

**Bring home** (Save Version, then download from the output): every
`*.csv`/`*.json` in `/kaggle/working/` → the repo's `results/` (update
`docs/experiments_results.md` first, `docs/paper_draft.md` second — the
sections marked "[pending re-run]"/"[numbers pending]").

## 1. Setup (same as 00/01/04 — see 00 for the full explanation)

In [ ]:
# Idempotent bootstrap -- safe to re-run (session restart, or you ran the cell twice).
# The old version was `!git clone` + `%cd`: it errored on the second run, and then
# left the notebook in the wrong directory with every relative path quietly broken.
import os, subprocess, sys

REPO_URL = "https://github.com/AIscend-Research/dental-extension.git"
if not os.path.exists("src/data/degradation.py"):          # not already at the repo root
    if not os.path.isdir("dental-extension"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir("dental-extension")
sys.path.insert(0, ".")

from src.utils.kaggle_env import install_deps, summarize_environment

# Installs ONLY what's missing, with numpy/torch pinned to the image's versions.
# Do NOT `pip install -r requirements-core.txt` here: it can upgrade numpy, and
# Kaggle's torch -- plus the detectron2 you are about to build against it -- is
# compiled for the numpy already in the image. The upgrade "succeeds", then torch
# dies at import with "compiled using NumPy 1.x cannot be run in NumPy 2.x".
print("installed:", install_deps() or "nothing needed -- image already has it")
for k, v in summarize_environment().items():
    print(f"  {k}: {v}")

# Build detectron2 against the image's torch. Never reinstall torch on Kaggle.
# Deliberately NOT -q: this compiles for ~10 minutes, and a silent cell that long
# is indistinguishable from a hang.
import torch
print(f"building detectron2 against torch {torch.__version__} -- expect ~10 min")

!pip install -q ninja
!pip install --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git"

!bash scripts/clone_baseline.sh  # import_hierarchicaldet needs external/HierarchicalDet to exist first

from src.utils.kaggle_env import import_hierarchicaldet

# Imports the real detectron2/pycocotools BEFORE putting HierarchicalDet on
# sys.path, so its vendored (uncompiled) copies cannot shadow them.
add_diffusiondet_config = import_hierarchicaldet()
print("hierarchialdet imports OK, using the real detectron2")

## 2. Locate the dataset, checkpoints, and head weights

Everything is found by a recursive glob under `/kaggle/input/` (or
`/kaggle/working/` if you're running in the same session that trained) — no
dataset slug to edit, and no assumption about whether Kaggle mounts inputs
flat (`/kaggle/input/<slug>/...`) or nested (`/kaggle/input/notebooks/<owner>/<slug>/...`).
A clear error tells you what to attach if something is missing.

In [ ]:
import glob
sys.path.insert(0, ".")
import cv2
import numpy as np
from detectron2.config import get_cfg
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import DatasetCatalog
from src.data.dentex import load_coco, patient_level_split, register_dentex_detectron2
from src.utils.kaggle_env import find_dentex_root

DATA_ROOT = str(find_dentex_root())
print("DATA_ROOT:", DATA_ROOT)

TRAINED_ARMS = ["baseline", "robustness"]


def find_output_file(rel):
    # recursive -- Kaggle nests attached notebook/dataset outputs under
    # /kaggle/input/notebooks/<owner>/<slug>/... or /kaggle/input/datasets/<owner>/<slug>/...,
    # not the flat /kaggle/input/<slug>/... some older accounts see.
    hits = sorted(glob.glob(f"/kaggle/input/**/{rel}", recursive=True)) + sorted(glob.glob(f"/kaggle/working/{rel}"))
    if not hits:
        raise FileNotFoundError(
            f"{rel} not found. Attach notebook 04's output as a data source "
            "(Add Data -> Your Work -> Notebook Output Files), or copy the file "
            "into /kaggle/working/."
        )
    if len(hits) > 1:
        print(f"note: multiple copies of {rel}; using {hits[0]}")
    return hits[0]


CHECKPOINTS = {arm: find_output_file(f"checkpoints_{arm}/model_final.pth") for arm in TRAINED_ARMS}
HEAD_WEIGHTS = {arm: find_output_file(f"confidence_head_{arm}.pth") for arm in TRAINED_ARMS}
for arm in TRAINED_ARMS:
    print(f"{arm}: {CHECKPOINTS[arm]}\n{' ' * len(arm)}  {HEAD_WEIGHTS[arm]}")

coco = load_coco(f"{DATA_ROOT}/train_quadrant_enumeration_disease.json")
split = patient_level_split(coco, seed=0)  # same seed as training -- val was truly held out
register_dentex_detectron2(coco, f"{DATA_ROOT}/xrays", split)
val_dicts = DatasetCatalog.get("custom_validation_class")
print("val images:", len(val_dicts))


def load_detector(arm):
    cfg = get_cfg()
    add_diffusiondet_config(cfg)
    cfg.merge_from_file("external/HierarchicalDet/configs/diffdet.custom.swinbase.nonpretrain.yaml")
    cfg.MODEL.WEIGHTS = CHECKPOINTS[arm]
    cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    model = build_model(cfg)
    DetectionCheckpointer(model).load(cfg.MODEL.WEIGHTS)
    model.eval()
    return cfg, model

## 3. Detection metrics per arm, across degradation severities

Notebook 03's verified inference loop (`k=2` — the default `k=0` silently
returns quadrant-level classes, see `kaggle/README.md`), generalized into a
function and swept over fixed severities. Two details that matter:

- **Ground truth is remapped through the same warp as the pixels**
  (`apply_degradations(..., boxes=...)`) — scoring degraded images against
  the original boxes would measure misalignment, not accuracy.
- **The degradation seed is the image id**, so every (arm, severity) cell of
  the table sees the *same* degraded images — a paired comparison, which is
  the point.

Sanity anchor from 03: an untrained checkpoint produces 0 predictions and
clean 0.0 metrics. Nonzero, severity-declining curves are the signal that
training worked; the robustness arm's curve staying above the baseline arm's
at high severity is claim #1 of the original plan.

In [ ]:
import csv, json, time
from src.eval.metrics import coco_map, per_class_f1
from src.data.degradation import apply_degradations

SEVERITIES = [0.0, 0.3, 0.5, 0.7]
TARGET = 800
CATEGORIES = [{"id": i, "name": n} for i, n in
              enumerate(["Impacted", "Caries", "Periapical Lesion", "Deep Caries"])]


def evaluate_arm(model, dicts, severity):
    predictions, gt_images, gt_annos = [], [], []
    ann_id = 0
    with torch.no_grad():
        for d in dicts:
            img = cv2.imread(d["file_name"], cv2.IMREAD_COLOR)
            annos = [a for a in d.get("annotations", []) if not a.get("iscrowd", 0)]
            boxes_xywh = np.array([a["bbox"] for a in annos], dtype=np.float64).reshape(-1, 4)

            if severity > 0:
                res = apply_degradations(img, severity_range=(severity, severity),
                                         boxes=boxes_xywh, seed=int(d["image_id"]))
                img, boxes_xywh = res.image, res.boxes
                keep = (boxes_xywh[:, 2] > 1.0) & (boxes_xywh[:, 3] > 1.0)
                boxes_xywh = boxes_xywh[keep]
                annos = [a for a, k in zip(annos, keep) if k]

            gt_images.append({"id": d["image_id"], "file_name": d["file_name"]})
            for a, bb in zip(annos, boxes_xywh):
                gt_annos.append({"id": ann_id, "image_id": d["image_id"],
                                 "category_id": a["category_id_3"],
                                 "bbox": [float(v) for v in bb], "iscrowd": 0,
                                 "area": float(bb[2] * bb[3])})
                ann_id += 1

            h0, w0 = img.shape[:2]
            img_r = cv2.resize(img, (TARGET, TARGET))
            inp = {"image": torch.as_tensor(img_r.transpose(2, 0, 1).astype(np.float32)),
                   "height": TARGET, "width": TARGET}
            out = model([inp], k=2)[0]["instances"]  # k=2 -> pred_classes_3 (diagnosis)
            sx, sy = w0 / TARGET, h0 / TARGET
            for box, score, cls in zip(out.pred_boxes.tensor.cpu().numpy(),
                                       out.scores.cpu().numpy(),
                                       out.pred_classes_3.cpu().numpy()):
                x1, y1, x2, y2 = box
                predictions.append({"image_id": d["image_id"], "category_id": int(cls),
                                    "bbox": [float(x1 * sx), float(y1 * sy),
                                             float((x2 - x1) * sx), float((y2 - y1) * sy)],
                                    "score": float(score)})
    gt = {"images": gt_images, "annotations": gt_annos, "categories": CATEGORIES}
    return predictions, gt


sweep_rows, f1_tables = [], {}
for arm in TRAINED_ARMS:
    cfg, model = load_detector(arm)
    for s in SEVERITIES:
        t0 = time.time()
        preds, gt = evaluate_arm(model, val_dicts, s)
        m = coco_map(preds, gt)
        row = {"arm": arm, "severity": s, "n_predictions": len(preds), **m}
        sweep_rows.append(row)
        print(f"[{arm} @ severity {s}] {m}  ({len(preds)} preds, {time.time()-t0:.0f}s)")
        if s in (0.0, SEVERITIES[-1]):
            f1_tables[f"{arm}_s{s}"] = per_class_f1(preds, gt)
    del model
    torch.cuda.empty_cache()

with open("/kaggle/working/detector_severity_sweep.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(sweep_rows[0]))
    w.writeheader()
    w.writerows(sweep_rows)
with open("/kaggle/working/detector_per_class_f1.json", "w") as f:
    json.dump(f1_tables, f, indent=2, default=str)
print("\nwrote /kaggle/working/detector_severity_sweep.csv and detector_per_class_f1.json")

## 4. Wire the real `DetectorChannel`

`KaggleCariesDetector` implements the `encode`/`predict` interface
`src/models/detector.py`'s stub documents (and `DetectorChannel` duck-types
against): `encode` → the FPN p5 feature map the confidence head was trained
on in notebook 04; `predict` → `k=2` detections as `{class_id, score}`
dicts. `caries_class_ids=(1, 3)` is Caries and Deep Caries in the diagnosis
task's `category_id_3` — locked by `docs/decisions.md` #1, not a choice made
here. The head is the **arm-matched** one: features from one arm's backbone
read by a head trained on another arm's features would be silently
miscalibrated.

In [ ]:
from src.models.detector_channel import DetectorChannel
from src.models.confidence_head import ConfidenceHead
from src.models.diagnostic import Case
from src.data.dentex_crops import load_tooth_crops, split_by_source_image, describe

CHANNEL_ARM = "robustness"  # the deployment-relevant arm; switch to compare
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class KaggleCariesDetector:
    """Real implementation of the CariesDetector interface (src/models/detector.py
    is still a stub in-repo; this is the wiring its TODOs describe)."""

    def __init__(self, cfg, model, target_size=800):
        self.model = model
        self.target_size = target_size
        self.device = cfg.MODEL.DEVICE
        self.pixel_mean = torch.tensor(cfg.MODEL.PIXEL_MEAN).view(1, 3, 1, 1).to(self.device)
        self.pixel_std = torch.tensor(cfg.MODEL.PIXEL_STD).view(1, 3, 1, 1).to(self.device)

    def _prep(self, image):
        img = np.asarray(image)
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        return cv2.resize(img, (self.target_size, self.target_size))

    def encode(self, image):
        img = self._prep(image).astype(np.float32)
        x = torch.from_numpy(img.transpose(2, 0, 1)[None]).to(self.device)
        x = (x - self.pixel_mean) / self.pixel_std
        return self.model.backbone(x)["p5"]

    def predict(self, image):
        img = self._prep(image)
        inp = {"image": torch.as_tensor(img.transpose(2, 0, 1).astype(np.float32)),
               "height": self.target_size, "width": self.target_size}
        out = self.model([inp], k=2)[0]["instances"]  # k=2: diagnosis-level classes
        return [{"class_id": int(c), "score": float(s)}
                for c, s in zip(out.pred_classes_3.cpu().numpy(), out.scores.cpu().numpy())]


cfg_ch, model_ch = load_detector(CHANNEL_ARM)
detector = KaggleCariesDetector(cfg_ch, model_ch)
head = ConfidenceHead(in_features=256)
head.load_state_dict(torch.load(HEAD_WEIGHTS[CHANNEL_ARM], map_location="cpu"))
head.to(DEVICE).eval()
channel = DetectorChannel(detector=detector, head=head, caries_class_ids=(1, 3))

# smoke test on one real crop before committing hours to the Docket
crops = load_tooth_crops(annotations=f"{DATA_ROOT}/train_quadrant_enumeration_disease.json",
                         image_root=f"{DATA_ROOT}/xrays", task="caries_vs_other")
rng = np.random.default_rng(0)
case = Case(label=crops[0].label, difficulty=0.0, payload=crops[0].image)
reading = channel.read(case, {"blur": 0.4}, rng)
print(f"smoke test OK -- score={reading.score:.3f}, usability={reading.usability:.3f}, "
      f"degradation={ {k: round(v, 2) for k, v in reading.degradation.items()} }")

## 5. The Docket, with a real trained detector

E6's protocol verbatim (`experiments/e6_real_images.py`, task B), with
`DetectorChannel` where E6 used its trained linear `RealImageChannel`: split
crops by source radiograph (same seed), calibrate on held-out first shots,
run every policy arm, print + save both leaderboards. Compare directly
against `results/e6_real_leaderboard_*.csv`.

Cost scales with `N_CASES × K` detector forwards — 3000 × 4 is the roadmap's
~12k-forward / 2–4 h estimate. The calibration pass prints its own timing
first; if it projects the Docket past your session budget, lower `N_CASES`
(the leaderboard gets noisier, not biased).

In [ ]:
from experiments.common import CLINIC_DIFFICULTY, HEADLINE_BURDEN
from experiments.e6_real_images import ARMS, collect_real_calibration
from src.bench.docket import make_image_docket
from src.bench.metrics import format_leaderboard, score_results
from src.bench.policies import policy_by_name
from src.bench.runner import run_docket
from src.evidence.calibration import LikelihoodRatioCalibrator, StratifiedCalibrator

N_CASES = 3000
BUDGET = 4
CAL_SHOTS_PER_CROP = 40

train_b, cal_crops, test = split_by_source_image(crops, seed=3)  # E6's exact split
print(describe(cal_crops, "calibration"))
print(describe(test, "test"))
print("(train split unused -- the detector arrives already trained, unlike E6's linear reader)")

t0 = time.time()
cal_data = collect_real_calibration(channel, cal_crops, rng, n_per_crop=CAL_SHOTS_PER_CROP)
cal_minutes = (time.time() - t0) / 60
per_read = (time.time() - t0) / max(len(cal_data.scores), 1)
print(f"calibration: {len(cal_data.scores)} first shots in {cal_minutes:.0f} min "
      f"({per_read:.2f}s/read -> Docket worst case ~{N_CASES * BUDGET * per_read / 3600:.1f}h)")

calibrator = LikelihoodRatioCalibrator(n_strata=4).fit(cal_data.scores, cal_data.labels, cal_data.usabilities)
conformal = StratifiedCalibrator(n_strata=4).fit(cal_data.scores, cal_data.labels, cal_data.usabilities)

pool = [c.image for c in test]
docket = make_image_docket("real_detector", [c.label for c in test], n_cases=N_CASES,
                           budget=BUDGET, burden=HEADLINE_BURDEN,
                           clinic_difficulty=CLINIC_DIFFICULTY, seed=5)
print(f"docket: {len(docket)} cases over {len(pool)} teeth, prevalence {docket.prevalence:.3f}, K={BUDGET}")

leaderboards = {}  # kept for the figure cell below
for label, cal in [("likelihood-ratio", calibrator), ("conformal", conformal)]:
    rows = [score_results(name,
                          run_docket(docket, policy_by_name(name), channel, cal, image_pool=pool),
                          docket.burden)
            for name in ARMS]
    leaderboards[label] = rows
    print(f"\nevidence construction: {label} (arm: {CHANNEL_ARM} detector)")
    print(format_leaderboard(rows))
    out = f"/kaggle/working/detector_leaderboard_{label}.csv"
    dicts = [r.as_dict() for r in rows]
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(dicts[0]))
        w.writeheader()
        w.writerows(dicts)
    print(f"wrote {out}")

## 6. The paper figure — `detector_robustness.png`

Two panels in the repo's house style (`experiments/e6_real_images.py`):

- **(a) the robustness claim as a picture** — mAP50 vs degradation severity,
  one line per arm, fixed colors (baseline blue, robustness orange — the
  pair is colorblind-safe, and identity is carried by the legend + direct
  labels, not color alone). The claim is the *gap between the lines at high
  severity*, so that gap is annotated directly.
- **(b) the Docket with the real detector** — verdicts-per-capture vs
  verdict accuracy, one point per policy, same marker convention as E6's
  panel (c): filled circle = guarantee held, open red X = violated. Compare
  side by side with `figures/e6_real_images.png`.

Labels are placed after `tight_layout()` — layout resizing moves the
data-to-pixel mapping under offset-anchored annotations, which silently
reintroduces the collisions `annotate_no_overlap` exists to prevent (E6
learned this the hard way; see its source comment).

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from experiments.common import annotate_no_overlap

# fixed color-by-entity: an arm keeps its color everywhere this project plots it
ARM_COLORS = {"baseline": "tab:blue", "robustness": "tab:orange"}

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6))

# --- (a) mAP50 vs severity, per arm --------------------------------------
ax = axes[0]
for arm in TRAINED_ARMS:
    rows = sorted((r for r in sweep_rows if r["arm"] == arm), key=lambda r: r["severity"])
    sev = [r["severity"] for r in rows]
    ax.plot(sev, [r["mAP50"] for r in rows], "o-", color=ARM_COLORS[arm], label=arm, ms=5)
    ax.annotate(arm, (sev[-1], rows[-1]["mAP50"]), xytext=(6, 0),
                textcoords="offset points", va="center", fontsize=8)
# annotate the gap at the highest severity -- the number the claim rests on
by_arm = {r["arm"]: r for r in sweep_rows if r["severity"] == SEVERITIES[-1]}
if len(by_arm) == 2:
    gap = by_arm["robustness"]["mAP50"] - by_arm["baseline"]["mAP50"]
    ax.annotate(f"gap {gap:+.3f}", (SEVERITIES[-1], np.mean([r["mAP50"] for r in by_arm.values()])),
                xytext=(-8, 0), textcoords="offset points", ha="right", fontsize=8)
ax.set_xlabel("degradation severity (paired images across arms)")
ax.set_ylabel("mAP50 on held-out DENTEX val")
ax.set_title("(a) robustness training vs capture degradation")
ax.set_ylim(bottom=0)
ax.margins(x=0.12)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- (b) the Docket, real detector (likelihood-ratio arm) ----------------
ax = axes[1]
board = leaderboards["likelihood-ratio"]
for r in board:
    ok = r.guaranteed and not (r.convict_violation or r.discharge_violation)
    ax.scatter(r.verdicts_per_capture, r.verdict_accuracy, s=90,
               facecolors="tab:blue" if ok else "none",
               edgecolors="tab:blue" if ok else "tab:red", linewidths=1.8,
               marker="o" if ok else "X")
ax.margins(0.18)
ax.set_xlabel("verdicts per capture")
ax.set_ylabel("accuracy on rendered verdicts")
ax.set_title(f"(b) The Docket, real {CHANNEL_ARM} detector\n(open red X = guarantee violated)")
ax.grid(alpha=0.3)
ax_b = ax

fig.suptitle(f"Real trained detector ({CHANNEL_ARM} arm channel), DENTEX", fontsize=12)
fig.tight_layout()
# placed after tight_layout() -- see the markdown note above / E6's source comment
annotate_no_overlap(
    ax_b, [r.verdicts_per_capture for r in board], [r.verdict_accuracy for r in board],
    [r.policy.replace("_", "\n") for r in board],
)
FIG_PATH = "/kaggle/working/detector_robustness.png"
fig.savefig(FIG_PATH, dpi=140)
plt.close(fig)
print(f"wrote {FIG_PATH}")

# render inline for a layout eyeball -- check for label collisions before trusting it
from IPython.display import Image as _Image, display as _display
_display(_Image(FIG_PATH))

## 7. Done — what goes back into the repo

Save Version, download from this version's output:

- `detector_severity_sweep.csv` + `detector_per_class_f1.json` → `results/`
  (the robustness table: baseline vs robustness mAP across severities)
- `detector_leaderboard_likelihood-ratio.csv` +
  `detector_leaderboard_conformal.csv` → `results/` (the real-detector Docket;
  compare against `e6_real_leaderboard_*.csv`)
- `detector_robustness.png` → `figures/` (the two-panel paper figure;
  eyeball it inline above first — the validator-checked part is color, not
  layout, so label collisions and clipped annotations are yours to catch)

Then update `docs/experiments_results.md` first and `docs/paper_draft.md`
second (its "[pending re-run]" / "[numbers pending]" sections) — the results
doc is canonical, the draft compresses it. If the two leaderboards
qualitatively disagree with E6's (e.g. the near-strong-reader pattern
flips), that is a finding to report, not to reconcile away.